# Pan-Cancer CAMP Metabolomics Integration

### Goal
Integrate an arbitrary Gene Signature with the pan-cancer mass spectrometry metabolomics dataset (CAMP, PMID: 37337120).

### Instructions
1. Edit the `TARGET_GENES` array below to define your gene signature.
2. Edit `TARGET_COHORTS` to restrict the analysis to specific TCGA-related cohorts if desired.
3. Run all cells to map the genes to their metabolites, merge with expression and TME data, and output the analysis.\n

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ==========================================
# PARAMETERS
# ==========================================
# Change these genes to your desired signature
TARGET_GENES = ["IDO1", "GDA", "CD38"]

# Which cohorts to load. 
# Available TCGA-like in CAMP: 'BRCA1', 'BRCA2', 'COAD', 'DLBCL', 'GBM', 'HCC', 'HurthleCC', 'ICC', 'OV', 'PDAC', 'PRAD', 'ccRCC1', 'ccRCC2', 'ccRCC3', 'ccRCC4'
# Leave as None to load all available inside the directory.
TARGET_COHORTS = ['BRCA1', 'BRCA2', 'COAD', 'OV', 'PRAD', 'GBM', 'PDAC']

# Paths
BASE_DIR = os.path.dirname(os.path.abspath('.')) # assumes this is run from scripts/
INPUT_DIR = os.path.join(BASE_DIR, 'input')
CAMP_DIR = os.path.join(INPUT_DIR, 'pancancer_metabolomics_2023_PMID37337120', 'data')
DB_PATH = os.path.join(INPUT_DIR, 'databases', 'human_database_merge_unique_metab_target_pairs_with_HMDB_Info.csv')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output', 'camp_integration')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Target Genes: {TARGET_GENES}")\n

In [ ]:
# ==========================================
# 1. LOAD DATABASE AND EXTRACT TARGET METABOLITES
# ==========================================
print("Loading database...")
# We use target_gene as per user preference
db = pd.read_csv(DB_PATH, low_memory=False)

# Filter for the target genes
sig_db = db[db['Target_Gene'].isin(TARGET_GENES)].copy()
signature_metabolites = sig_db['Metabolite_Name'].dropna().unique().tolist()

print(f"Found {len(signature_metabolites)} metabolites associated with the target genes:")
for g in TARGET_GENES:
    assoc_mets = sig_db[sig_db['Target_Gene'] == g]['Metabolite_Name'].dropna().unique()
    print(f"  - {g}: {len(assoc_mets)} metabolites -> {', '.join(assoc_mets[:5])}...")\n

In [ ]:
# ==========================================
# 2. DATA LOADING LOOP (Transcriptomics, Metabolomics, TME)
# ==========================================

metab_dir = os.path.join(CAMP_DIR, 'metabolomics_processed')
trans_dir = os.path.join(CAMP_DIR, 'transcriptomics_processed')
tme_dir = os.path.join(CAMP_DIR, 'TME_deconvolution_processed')

all_cohorts_data = []

# If no target cohorts are specified, grab all Excel files in the metabolomics directory
if TARGET_COHORTS is None:
    TARGET_COHORTS = [f.replace('PreprocessedData_', '').replace('.xlsx', '') 
                      for f in os.listdir(metab_dir) if f.endswith('.xlsx')]

print(f"Processing Cohorts: {TARGET_COHORTS}")

for cohort in TARGET_COHORTS:
    metab_file = os.path.join(metab_dir, f"PreprocessedData_{cohort}.xlsx")
    if not os.path.exists(metab_file):
        print(f"Skipping {cohort} - no metabolomics file found.")
        continue
    
    # 2a. Load Metabolomics
    # Structure of CAMP metabolomics excel varies slightly but typically has sample IDs in rows/cols
    # Will attempt to read and transpose if necessary so Samples are index
    df_metab = pd.read_excel(metab_file, index_col=0)
    
    # Check if samples are rows or columns. Often in CAMP, metabolites are columns or rows.
    # We will assume if there are thousands of features it's transposed.
    if df_metab.shape[0] < df_metab.shape[1]:
        # Samples are likely rows
        pass
    else:
        df_metab = df_metab.T

    # Keep only target metabolites
    keep_metabs = [m for m in df_metab.columns if m in signature_metabolites]
    df_metab = df_metab[keep_metabs]
    
    # Rename columns to indicate these are metabolites
    df_metab.columns = [f"METAB_{c}" for c in df_metab.columns]
    
    # 2b. Load Transcriptomics (TPM)
    trans_file = os.path.join(trans_dir, [f for f in os.listdir(trans_dir) if cohort in f and 'tpm' in f][0]) if any(cohort in f and 'tpm' in f for f in os.listdir(trans_dir)) else None
    
    if trans_file:
        df_trans = pd.read_csv(trans_file, index_col=0)
        if df_trans.shape[0] < df_trans.shape[1]:
            pass
        else:
            df_trans = df_trans.T
            
        keep_genes = [g for g in df_trans.columns if g in TARGET_GENES]
        df_trans = df_trans[keep_genes]
        df_trans.columns = [f"GENE_{c}" for c in df_trans.columns]
    else:
        df_trans = pd.DataFrame(index=df_metab.index)
        print(f"  Warning: No TPM transcriptomics found for {cohort}")

    # 2c. Load TME Deconvolution
    tme_file = os.path.join(tme_dir, [f for f in os.listdir(tme_dir) if cohort in f][0]) if any(cohort in f for f in os.listdir(tme_dir)) else None
    if tme_file:
        df_tme = pd.read_csv(tme_file, index_col=0)
        df_tme.columns = [f"TME_{c}" for c in df_tme.columns]
    else:
        df_tme = pd.DataFrame(index=df_metab.index)
        
    # Merge for this cohort
    # We assume the indices are roughly matched or overlapping Sample IDs
    # Sometimes Normal samples have '-N' or 'N' suffix. We will extract Tumor/Normal status
    df_cohort = df_metab.join(df_trans, how='outer').join(df_tme, how='outer')
    df_cohort['Cohort'] = cohort
    
    # Naive extraction of Tumor/Normal from index (typically CAMP designates Normal as adjacent)
    # This may need adjustment based on exact naming conventions in MasterMapping
    df_cohort['Tissue_Type'] = df_cohort.index.map(lambda x: 'Normal' if str(x).endswith('N') or '_N' in str(x) else 'Tumor')
    
    all_cohorts_data.append(df_cohort)

# Combine all
master_df = pd.concat(all_cohorts_data)
print(f"\nSuccessfully assembled multi-omics master table: {master_df.shape[0]} samples, {master_df.shape[1]} features.")\n

In [ ]:
# ==========================================
# 3. TUMOR vs NORMAL ANALYSIS
# ==========================================
# Let's plot Differential Abundance / Expression for the signature targets
tumor_normal_df = master_df.dropna(subset=[c for c in master_df.columns if c.startswith('METAB_') or c.startswith('GENE_')], how='all')

features_to_plot = [c for c in master_df.columns if c.startswith('METAB_') or c.startswith('GENE_')]

if len(features_to_plot) > 0 and len(tumor_normal_df['Tissue_Type'].unique()) > 1:
    fig, axes = plt.subplots(len(features_to_plot), 1, figsize=(10, 4*len(features_to_plot)))
    if len(features_to_plot) == 1: axes = [axes]
    
    for i, feature in enumerate(features_to_plot):
        sns.boxplot(data=tumor_normal_df, x='Cohort', y=feature, hue='Tissue_Type', ax=axes[i])
        axes[i].set_title(f"Tumor vs Normal: {feature}")
        axes[i].tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'tumor_vs_normal_boxplots.png'))
    plt.show()
else:
    print("Not enough overlapping Tumor/Normal labels or features to plot.")\n

In [ ]:
# ==========================================
# 4. GENE-METABOLITE COVARIATION (CONCORDANCE)
# ==========================================
# We compute Spearman correlation across all tumors
tumor_df = master_df[master_df['Tissue_Type'] == 'Tumor']

gene_cols = [c for c in tumor_df.columns if c.startswith('GENE_')]
metab_cols = [c for c in tumor_df.columns if c.startswith('METAB_')]

if gene_cols and metab_cols:
    corr_matrix = pd.DataFrame(index=gene_cols, columns=metab_cols)
    pval_matrix = pd.DataFrame(index=gene_cols, columns=metab_cols)

    for g in gene_cols:
        for m in metab_cols:
            valid = tumor_df[[g, m]].dropna()
            if len(valid) > 10:
                rho, p = stats.spearmanr(valid[g], valid[m])
                corr_matrix.loc[g, m] = rho
                pval_matrix.loc[g, m] = p

    # Plot Heatmap
    corr_matrix = corr_matrix.astype(float)
    plt.figure(figsize=(10, max(4, len(gene_cols)*0.5)))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, vmin=-1, vmax=1)
    plt.title("Gene-Metabolite Spearman Correlation (Tumors only)")
    plt.savefig(os.path.join(OUTPUT_DIR, 'gene_metabolite_covariation_heatmap.png'))
    plt.show()\n

In [ ]:
# ==========================================
# 5. METABOLITE - TME (IMMUNE CELL) ASSOCIATIONS
# ==========================================
# The paper showed hub metabolites (like NAD+) associate heavily with immune cell signatures
tme_cols = [c for c in tumor_df.columns if c.startswith('TME_')]

# Let's pick a subset of TME columns, maybe just specific immune cells like Mast cells, Macrophages, Dendritic cells
immune_cells = [c for c in tme_cols if any(x in c.lower() for x in ['mast', 'macrophage', 'dc', 'dendritic', 't cell', 'b cell', 'nk'])]

if metab_cols and immune_cells:
    tme_corr = pd.DataFrame(index=metab_cols, columns=immune_cells)
    
    for m in metab_cols:
        for tme in immune_cells:
            valid = tumor_df[[m, tme]].dropna()
            if len(valid) > 10:
                rho, p = stats.spearmanr(valid[m], valid[tme])
                tme_corr.loc[m, tme] = rho

    tme_corr = tme_corr.astype(float)
    plt.figure(figsize=(12, max(5, len(metab_cols)*0.8)))
    sns.heatmap(tme_corr, annot=False, cmap='PRGn', center=0) # without annot if too large
    plt.title("Metabolite vs Immune Cell Population Correlation")
    plt.savefig(os.path.join(OUTPUT_DIR, 'metabolite_immune_covariation_heatmap.png'))
    plt.show()\n

In [ ]:
# Save Master Data to Output
output_csv = os.path.join(OUTPUT_DIR, 'master_integrated_camp_data.csv')
master_df.to_csv(output_csv)
print(f"Analysis complete. Full merged data saved to {output_csv}")\n